# Loan Approved – Cleaning & Feature Engineering 

Every step changes the **same `df`** in place. Pattern used in each step:

1. `before = df.copy()` → save the state
2. change `df`
3. show **Before vs After**

| # | Step |
|---|---|
| 1 | Duplicates |
| 2 | Missing values |
| 3 | Fix data type (`Dependents`) |
| 4 | Outliers (IQR capping) |
| 5 | Skewness → log transform |
| 6 | Scaling |
| 7 | Encoding |
| 8 | Drop `Loan_ID` & final df |

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)   # show all columns
pd.set_option('display.width', 200)

In [2]:
df = pd.read_csv("loan_approved.csv")
df.rename(columns={'Loan_Status (Approved)': 'Loan_Status'}, inplace=True)   # remove spaces/brackets from name
df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            614 non-null    object 
 1   Gender             601 non-null    object 
 2   Married            611 non-null    object 
 3   Dependents         599 non-null    object 
 4   Education          614 non-null    object 
 5   Self_Employed      582 non-null    object 
 6   ApplicantIncome    614 non-null    int64  
 7   CoapplicantIncome  614 non-null    float64
 8   LoanAmount         592 non-null    float64
 9   Loan_Amount_Term   600 non-null    float64
 10  Credit_History     564 non-null    float64
 11  Property_Area      614 non-null    object 
 12  Loan_Status        614 non-null    object 
dtypes: float64(4), int64(1), object(8)
memory usage: 62.5+ KB


## 1. Duplicates

**When:** after merging files / joins. Duplicate rows over-weight some records.

In [4]:
before = df.copy()
print("Duplicates before:", df.duplicated().sum(), "| shape:", df.shape)

df.drop_duplicates(inplace=True)

print("Duplicates after :", df.duplicated().sum(), "| shape:", df.shape)

Duplicates before: 0 | shape: (614, 13)
Duplicates after : 0 | shape: (614, 13)


No duplicates here, so nothing changed – but the check is always needed.

## 2. Missing values

| Column | Fill with | Why |
|---|---|---|
| `Gender`, `Married`, `Self_Employed`, `Dependents` | most common value (mode) | categorical |
| `LoanAmount` | median | skewed, has outliers → mean is pulled up |
| `Credit_History` | median (= mode, 1.0) | 0/1 flag |
| `Loan_Amount_Term` | 360 (mode) | most loans are 360 months |

**Use `dropna()` only when very few rows are null** – here it would delete ~22% of rows.

In [5]:
before = df.copy()
print(before.isnull().sum())

Loan_ID               0
Gender               13
Married               3
Dependents           15
Education             0
Self_Employed        32
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount           22
Loan_Amount_Term     14
Credit_History       50
Property_Area         0
Loan_Status           0
dtype: int64


In [6]:
df.fillna(value = {'Gender': 'Male',
                   'LoanAmount': df.LoanAmount.median(),
                   'Credit_History': df.Credit_History.median(),
                   'Dependents': '0',
                   'Self_Employed': 'No',
                   'Loan_Amount_Term': 360.0}, inplace=True)

df.loc[df.Married.isnull(), 'Married'] = 'Yes'

In [7]:
# DIFF: null count before vs after
pd.DataFrame({'Before': before.isnull().sum(), 'After': df.isnull().sum()})

,Before,After
Loan_ID,0,0
Gender,13,0
Married,3,0
Dependents,15,0
Education,0,0
Self_Employed,32,0
ApplicantIncome,0,0
CoapplicantIncome,0,0
LoanAmount,22,0
Loan_Amount_Term,14,0


In [8]:
# DIFF: rows that had nulls (Before)
rows = before.isnull().any(axis=1)
before[rows].head(8)

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
11,LP001027,Male,Yes,2,Graduate,NaN,2500,1840.0,109.0,360.0,1.0,Urban,Y
16,LP001034,Male,No,1,Not Graduate,No,3596,0.0,100.0,240.0,NaN,Urban,Y
19,LP001041,Male,Yes,0,Graduate,NaN,2600,3500.0,115.0,NaN,1.0,Urban,Y
23,LP001050,NaN,Yes,2,Not Graduate,No,3365,1917.0,112.0,360.0,0.0,Rural,N
24,LP001052,Male,Yes,1,Graduate,NaN,3717,2925.0,151.0,360.0,NaN,Semiurban,N
29,LP001087,Female,No,2,Graduate,NaN,3750,2083.0,120.0,360.0,1.0,Semiurban,Y
30,LP001091,Male,Yes,1,Graduate,NaN,4166,3369.0,201.0,360.0,NaN,Urban,N


In [9]:
# DIFF: same rows (After)
df[rows].head(8)

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,128.0,360.0,1.0,Urban,Y
11,LP001027,Male,Yes,2,Graduate,No,2500,1840.0,109.0,360.0,1.0,Urban,Y
16,LP001034,Male,No,1,Not Graduate,No,3596,0.0,100.0,240.0,1.0,Urban,Y
19,LP001041,Male,Yes,0,Graduate,No,2600,3500.0,115.0,360.0,1.0,Urban,Y
23,LP001050,Male,Yes,2,Not Graduate,No,3365,1917.0,112.0,360.0,0.0,Rural,N
24,LP001052,Male,Yes,1,Graduate,No,3717,2925.0,151.0,360.0,1.0,Semiurban,N
29,LP001087,Female,No,2,Graduate,No,3750,2083.0,120.0,360.0,1.0,Semiurban,Y
30,LP001091,Male,Yes,1,Graduate,No,4166,3369.0,201.0,360.0,1.0,Urban,N


## 3. Fix data type – `Dependents`

`Dependents` has the text `'3+'`, so the column is a string. Convert it to a number (`3` = 3 or more).
Done **after** filling nulls (int cannot hold NaN).

In [10]:
before = df.copy()

df['Dependents'] = df['Dependents'].replace('3+', '3').astype(int)
df['Credit_History'] = df['Credit_History'].astype(int)
df['Loan_Amount_Term'] = df['Loan_Amount_Term'].astype(int)

In [11]:
# DIFF: dtype before vs after
pd.DataFrame({'Before': before.dtypes.astype(str), 'After': df.dtypes.astype(str)})

,Before,After
Loan_ID,object,object
Gender,object,object
Married,object,object
Dependents,object,int32
Education,object,object
Self_Employed,object,object
ApplicantIncome,int64,int64
CoapplicantIncome,float64,float64
LoanAmount,float64,float64
Loan_Amount_Term,float64,int32


In [12]:
# DIFF: values of Dependents
print("Before:", sorted(before.Dependents.unique()))
print("After :", sorted(df.Dependents.unique().tolist()))

Before: ['0', '1', '2', '3+']
After : [0, 1, 2, 3]


## 4. Outliers – IQR method (capping)

Values below `Q1 - 1.5*IQR` or above `Q3 + 1.5*IQR` are replaced by the bound (rows are **not** deleted).

**When:** skewed numeric data (income, amounts). For roughly normal data use mean ± 3·std. Tree models can keep outliers.

In [13]:
before = df.copy()

df[['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount']] = df[['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount']].astype(float)   # bounds are decimals, so int columns -> float

for col in ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    print(col, "-> Lower:", lower_bound, "Upper:", upper_bound)

    df.loc[df[col] < lower_bound, col] = lower_bound
    df.loc[df[col] > upper_bound, col] = upper_bound

ApplicantIncome -> Lower: -1498.75 Upper: 10171.25
CoapplicantIncome -> Lower: -3445.875 Upper: 5743.125
LoanAmount -> Lower: 3.5 Upper: 261.5


In [14]:
# DIFF: summary before vs after
cols = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount']
print("BEFORE"); print(before[cols].describe().round(2))
print("AFTER");  print(df[cols].describe().round(2))

BEFORE
       ApplicantIncome  CoapplicantIncome  LoanAmount
count           614.00             614.00      614.00
mean           5403.46            1621.25      145.75
std            6109.04            2926.25       84.11
min             150.00               0.00        9.00
25%            2877.50               0.00      100.25
50%            3812.50            1188.50      128.00
75%            5795.00            2297.25      164.75
max           81000.00           41667.00      700.00
AFTER
       ApplicantIncome  CoapplicantIncome  LoanAmount
count           614.00             614.00      614.00
mean           4617.11            1419.70      137.37
std            2479.85            1624.61       55.78
min             150.00               0.00        9.00
25%            2877.50               0.00      100.25
50%            3812.50            1188.50      128.00
75%            5795.00            2297.25      164.75
max           10171.25            5743.12      261.50


In [15]:
# DIFF: only the rows that were capped (LoanAmount example)
changed = before.LoanAmount != df.LoanAmount
pd.DataFrame({'Before': before.LoanAmount[changed], 'After': df.LoanAmount[changed]}).head(8)

,Before,After
5,267.0,261.5
9,349.0,261.5
21,315.0,261.5
34,320.0,261.5
54,286.0,261.5
67,312.0,261.5
83,265.0,261.5
126,370.0,261.5


In [16]:
# DIFF: how many values were capped per column
for col in cols:
    print(col, ":", (before[col] != df[col]).sum(), "values capped")

ApplicantIncome : 50 values capped
CoapplicantIncome : 18 values capped
LoanAmount : 41 values capped


## 5. Skewness → Log transformation

Skew > 0 = right skewed. `np.log1p` = log(1 + x) → works even when value is 0 (`CoapplicantIncome` has many 0s; plain `np.log(0)` = -inf).

**When:** right-skewed positive data (income, price). Moderate skew → square root; any sign → Yeo-Johnson.

In [17]:
before = df.copy()
print("Skew BEFORE")
print(df[['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount']].skew())

Skew BEFORE
ApplicantIncome      1.039846
CoapplicantIncome    1.012763
LoanAmount           0.683693
dtype: float64


In [18]:
df['ApplicantIncome'] = np.log1p(df['ApplicantIncome'])
df['CoapplicantIncome'] = np.log1p(df['CoapplicantIncome'])
df['LoanAmount'] = np.log1p(df['LoanAmount'])

print("Skew AFTER")
print(df[['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount']].skew())

Skew AFTER
ApplicantIncome     -0.592116
CoapplicantIncome   -0.184932
LoanAmount          -0.896386
dtype: float64


In [19]:
# DIFF: first rows before vs after
print("BEFORE"); print(before[['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount']].head())
print("AFTER");  print(df[['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount']].head())

BEFORE
   ApplicantIncome  CoapplicantIncome  LoanAmount
0           5849.0                0.0       128.0
1           4583.0             1508.0       128.0
2           3000.0                0.0        66.0
3           2583.0             2358.0       120.0
4           6000.0                0.0       141.0
AFTER
   ApplicantIncome  CoapplicantIncome  LoanAmount
0         8.674197           0.000000    4.859812
1         8.430327           7.319202    4.859812
2         8.006701           0.000000    4.204693
3         7.857094           7.765993    4.795791
4         8.699681           0.000000    4.955827


## 6. Scaling – StandardScaler

Puts columns on the same scale: mean 0, std 1. Needed for distance/gradient models (KNN, SVM, Logistic Regression, PCA).
Not needed for tree models.

| Scaler | Use when |
|---|---|
| StandardScaler | roughly normal data (used here, after log transform) |
| MinMaxScaler | need values between 0 and 1, no outliers |
| RobustScaler | many outliers remain |

> In a real model, fit the scaler on the **train set only** to avoid data leakage.

In [20]:
before = df.copy()

scale_cols = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term']
df[scale_cols] = StandardScaler().fit_transform(df[scale_cols])

In [21]:
# DIFF: mean and std before vs after
pd.DataFrame({'Mean before': before[scale_cols].mean(), 'Mean after': df[scale_cols].mean().round(3),
              'Std before': before[scale_cols].std(), 'Std after': df[scale_cols].std().round(3)})

,Mean before,Mean after,Std before,Std after
ApplicantIncome,8.298615,0.0,0.545307,1.001
CoapplicantIncome,4.272817,-0.0,3.853723,1.001
LoanAmount,4.842624,0.0,0.440406,1.001
Loan_Amount_Term,342.410423,0.0,64.428629,1.001


In [22]:
# DIFF: first rows
print("BEFORE"); print(before[scale_cols].head())
print("AFTER");  print(df[scale_cols].head())

BEFORE
   ApplicantIncome  CoapplicantIncome  LoanAmount  Loan_Amount_Term
0         8.674197           0.000000    4.859812               360
1         8.430327           7.319202    4.859812               360
2         8.006701           0.000000    4.204693               360
3         7.857094           7.765993    4.795791               360
4         8.699681           0.000000    4.955827               360
AFTER
   ApplicantIncome  CoapplicantIncome  LoanAmount  Loan_Amount_Term
0         0.689316          -1.109654    0.039060          0.273231
1         0.241735           0.791149    0.039060          0.273231
2        -0.535757          -1.109654   -1.449687          0.273231
3        -0.810334           0.907181   -0.106428          0.273231
4         0.736088          -1.109654    0.257252          0.273231


## 7. Encoding categorical columns

| Method | Use when | Columns |
|---|---|---|
| Label encoding (0/1) | only 2 categories | `Gender`, `Married`, `Education`, `Self_Employed`, `Loan_Status` |
| One-Hot (`get_dummies`) | 3+ categories with **no order** | `Property_Area` |
| Ordinal encoding | categories have a real order (e.g. Low < Medium < High) | – |

In [23]:
before = df.copy()

# Label encoding (0/1) -> columns with only 2 categories
for col in ['Gender', 'Married', 'Education', 'Self_Employed', 'Loan_Status']:
    enc = LabelEncoder()                      # new encoder per column
    df[col] = enc.fit_transform(df[col])      # LabelEncoder works on a 1-D Series
    print(col, {k: int(v) for k, v in zip(enc.classes_, enc.transform(enc.classes_))})

Gender {'Female': 0, 'Male': 1}
Married {'No': 0, 'Yes': 1}
Education {'Graduate': 0, 'Not Graduate': 1}
Self_Employed {'No': 0, 'Yes': 1}
Loan_Status {'N': 0, 'Y': 1}


In [24]:
# One-Hot encoding (get_dummies) -> 3 categories, no order
df = pd.get_dummies(df, columns=['Property_Area'], dtype=int)
df.filter(like='Property_Area').head()

,Property_Area_Rural,Property_Area_Semiurban,Property_Area_Urban
0,0,0,1
1,1,0,0
2,0,0,1
3,0,0,1
4,0,0,1


> **Note:** If the categories had a real order, use `OrdinalEncoder` instead, e.g.
> `OrdinalEncoder(categories=[['Low','Medium','High']])`. `Property_Area` has no natural order, so One-Hot is used above.

In [25]:
# DIFF: columns df[['Gender','Gender_enc']before vs after
print("Before:", list(before.columns))
print("After :", list(df.columns))

Before: ['Loan_ID', 'Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History', 'Property_Area', 'Loan_Status']
After : ['Loan_ID', 'Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History', 'Loan_Status', 'Property_Area_Rural', 'Property_Area_Semiurban', 'Property_Area_Urban']


In [26]:
# DIFF: first rows
print("BEFORE"); print(before[['Gender', 'Married', 'Education', 'Self_Employed', 'Property_Area', 'Loan_Status']].head())
print("AFTER");  print(df[['Gender', 'Married', 'Education','Self_Employed']])
print(df[[ 'Property_Area_Rural', 'Property_Area_Semiurban', 'Property_Area_Urban', 'Loan_Status']].head())

BEFORE
  Gender Married     Education Self_Employed Property_Area Loan_Status
0   Male      No      Graduate            No         Urban           Y
1   Male     Yes      Graduate            No         Rural           N
2   Male     Yes      Graduate           Yes         Urban           Y
3   Male     Yes  Not Graduate            No         Urban           Y
4   Male      No      Graduate            No         Urban           Y
AFTER
     Gender  Married  Education  Self_Employed
0         1        0          0              0
1         1        1          0              0
2         1        1          0              1
3         1        1          1              0
4         1        0          0              0
..      ...      ...        ...            ...
609       0        0          0              0
610       1        1          0              0
611       1        1          0              0
612       1        1          0              0
613       0        0          0             

## 8. Drop identifier & final clean DataFrame

`Loan_ID` is unique for every row, so it has no predictive value.

In [27]:
before = df.copy()
print("Duplicate Loan_ID:", df.Loan_ID.duplicated().sum())
df.drop('Loan_ID', axis=1, inplace=True)
print("Shape before:", before.shape, "| after:", df.shape)

Duplicate Loan_ID: 0
Shape before: (614, 15) | after: (614, 14)


In [28]:
df.head(10)

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Loan_Status,Property_Area_Rural,Property_Area_Semiurban,Property_Area_Urban
0,1,0,0,0,0,0.689316,-1.109654,0.039060,0.273231,1,1,0,0,1
1,1,1,1,0,0,0.241735,0.791149,0.039060,0.273231,1,0,1,0,0
2,1,1,0,0,1,-0.535757,-1.109654,-1.449687,0.273231,1,1,0,0,1
3,1,1,0,1,0,-0.810334,0.907181,-0.106428,0.273231,1,1,0,0,1
4,1,0,0,0,0,0.736088,-1.109654,0.257252,0.273231,1,1,0,0,1
5,1,1,2,0,1,0.548519,1.056803,1.653519,0.273231,1,1,0,0,1
6,1,1,0,1,0,-0.997088,0.792522,-0.632377,0.273231,1,1,0,0,1
7,1,1,3,0,0,-0.513871,0.922776,0.514217,0.273231,0,0,0,1,0
8,1,1,2,0,0,-0.005169,0.794228,0.652826,0.273231,1,1,0,0,1
9,1,1,1,0,0,1.704657,1.138299,1.653519,0.273231,1,0,0,1,0


In [29]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Gender                   614 non-null    int32  
 1   Married                  614 non-null    int32  
 2   Dependents               614 non-null    int32  
 3   Education                614 non-null    int32  
 4   Self_Employed            614 non-null    int32  
 5   ApplicantIncome          614 non-null    float64
 6   CoapplicantIncome        614 non-null    float64
 7   LoanAmount               614 non-null    float64
 8   Loan_Amount_Term         614 non-null    float64
 9   Credit_History           614 non-null    int32  
 10  Loan_Status              614 non-null    int32  
 11  Property_Area_Rural      614 non-null    int32  
 12  Property_Area_Semiurban  614 non-null    int32  
 13  Property_Area_Urban      614 non-null    int32  
dtypes: float64(4), int32(10)
m

In [30]:
print("Nulls :", df.isnull().sum().sum())
print("Shape :", df.shape)

Nulls : 0
Shape : (614, 14)


In [31]:
df.to_csv("loan_approved_final.csv", index=False)